## Download the packages

In [ ]:
!pip install nltk gensim pyLDAvis

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 21.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import gensim
from gensim import corpora
from gensim.models import LdaModel
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
import matplotlib.pyplot as plt


In [ ]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

## Upload the CSV file

In [ ]:
# Step 2: Upload your CSV file
from google.colab import files
uploaded = files.upload()  # Use the file upload dialog to upload your CSV file

Saving data_filter_add.csv to data_filter_add.csv


In [ ]:
# Step 3: Load your dataset
# Replace 'your_dataset.csv' with the actual filename of your uploaded CSV
import io
df = pd.read_csv(io.BytesIO(uploaded['data_filter_add.csv']))

## Process the text

In [ ]:
# Step 4: Preprocess the text
# Define a function to clean and tokenize your text data
def preprocess(text):
    # Convert to lowercase
    text = text.lower()
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize the text
    tokens = word_tokenize(text)
    # Remove English stopwords
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    # Optionally, remove short tokens
    tokens = [word for word in tokens if len(word) > 2]
    return tokens


In [ ]:
# Assuming your CSV has a column named 'comment'
df['tokens'] = df['comment'].astype(str).apply(preprocess)

## Create dictionary and corpus for the topic modeling

In [ ]:
# Step 5: Create a dictionary and corpus needed for topic modeling
dictionary = corpora.Dictionary(df['tokens'])
# Filter out extreme cases (words that appear too rarely or too frequently)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in df['tokens']]

In [ ]:
# Step 6: Build the LDA model
# Set the number of topics (adjust as needed)
num_topics = 6

lda_model = LdaModel(corpus=corpus,
                     id2word=dictionary,
                     num_topics=num_topics,
                     random_state=42,
                     update_every=1,
                     chunksize=100,
                     passes=10,
                     alpha='auto',
                     per_word_topics=True)

# Print out the topics and their top words
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx}:")
    print(topic)
    print()


Topic 0:
0.065*"beautiful" + 0.038*"much" + 0.035*"nice" + 0.033*"really" + 0.029*"lol" + 0.026*"get" + 0.024*"know" + 0.021*"videos" + 0.017*"ive" + 0.014*"feel"

Topic 1:
0.059*"property" + 0.047*"place" + 0.029*"buy" + 0.027*"money" + 0.020*"could" + 0.020*"even" + 0.018*"area" + 0.018*"better" + 0.018*"make" + 0.017*"right"

Topic 2:
0.097*"house" + 0.048*"home" + 0.042*"would" + 0.035*"love" + 0.035*"one" + 0.026*"million" + 0.018*"live" + 0.017*"see" + 0.015*"room" + 0.015*"price"

Topic 3:
0.043*"beach" + 0.034*"island" + 0.019*"rich" + 0.018*"years" + 0.017*"new" + 0.017*"damn" + 0.017*"built" + 0.016*"next" + 0.014*"youre" + 0.014*"top"

Topic 4:
0.100*"like" + 0.049*"enes" + 0.040*"amazing" + 0.031*"thank" + 0.028*"good" + 0.028*"dont" + 0.027*"great" + 0.023*"looks" + 0.022*"video" + 0.020*"thanks"

Topic 5:
0.035*"homes" + 0.032*"look" + 0.030*"people" + 0.030*"view" + 0.024*"ocean" + 0.018*"many" + 0.017*"well" + 0.016*"pretty" + 0.016*"need" + 0.016*"didnt"



In [ ]:
# Step 7: Visualize the topics with pyLDAvis
pyLDAvis.enable_notebook()
vis = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(vis)

## Comparing Topic Prevalence by Property Type & Creator

In [ ]:
# Assuming you have your final LDA model with 6 topics stored in lda_model_6
lda_model_6 = lda_model  # or whatever variable holds your final 6-topic model

topic_dist = []
for i, bow in enumerate(corpus):
    # Get topic probabilities for each document
    doc_topics = lda_model_6.get_document_topics(bow)  # returns list of (topic_num, prob)

    # Build a dictionary to store info about this document
    row_dict = {
        'doc_id': i,
        'housetype': df.loc[i, 'housetype'],
        'creator': df.loc[i, 'creator']
    }

    # Initialize each topic proportion to 0.0
    for t in range(num_topics):
        row_dict[f"Topic_{t}"] = 0.0

    # Fill in the actual probabilities
    for (topic_num, prob) in doc_topics:
        row_dict[f"Topic_{topic_num}"] = prob

    topic_dist.append(row_dict)


In [ ]:
# Create a DataFrame with topic distributions + metadata
topics_df = pd.DataFrame(topic_dist)

In [ ]:
##############################################
# 2) Compare means by property_type
##############################################
property_group = topics_df.groupby('housetype').mean(numeric_only=True)
print("Average Topic Distribution by Property Type:")
print(property_group[[f"Topic_{t}" for t in range(num_topics)]])
print()

# Example interpretation:
# If "Topic_0" has mean=0.25 for beach and 0.10 for city,
# that suggests beach videos average a 25% share of words for Topic_0,
# while city videos average 10%.


Average Topic Distribution by Property Type:
            Topic_0   Topic_1   Topic_2   Topic_3   Topic_4   Topic_5
housetype                                                            
beach      0.151054  0.147635  0.277706  0.097870  0.189089  0.136646
city       0.152711  0.149213  0.269514  0.099655  0.188944  0.139963



In [ ]:
##############################################
# 3) Compare means by creator
##############################################
creator_group = topics_df.groupby('creator').mean(numeric_only=True)
print("Average Topic Distribution by Creator:")
print(creator_group[[f"Topic_{t}" for t in range(num_topics)]])
print()


Average Topic Distribution by Creator:
          Topic_0   Topic_1   Topic_2   Topic_3   Topic_4   Topic_5
creator                                                            
Enes     0.152674  0.148636  0.276202  0.097262  0.186911  0.138315
Ryan     0.150923  0.148225  0.269787  0.100827  0.191781  0.138457



In [ ]:
##############################################
# 4) Identify which topic is most common
#    for each property_type and each creator
##############################################
for prop_type, group_data in topics_df.groupby('housetype'):
    avg_distribution = group_data[[f"Topic_{t}" for t in range(num_topics)]].mean()
    top_topic = avg_distribution.idxmax()  # e.g., 'Topic_3'
    print(f"For property_type={prop_type}, the most common topic is {top_topic} with mean proportion {avg_distribution.max():.2f}")

for cr, group_data in topics_df.groupby('creator'):
    avg_distribution = group_data[[f"Topic_{t}" for t in range(num_topics)]].mean()
    top_topic = avg_distribution.idxmax()
    print(f"For creator={cr}, the most common topic is {top_topic} with mean proportion {avg_distribution.max():.2f}")

For property_type=beach, the most common topic is Topic_2 with mean proportion 0.28
For property_type=city, the most common topic is Topic_2 with mean proportion 0.27
For creator=Enes, the most common topic is Topic_2 with mean proportion 0.28
For creator=Ryan, the most common topic is Topic_2 with mean proportion 0.27


## Statistical Evidence

In [ ]:
import pandas as pd
from scipy.stats import ttest_ind

num_topics = 6

In [ ]:
##############################################
# 1) T-tests for "beach" vs. "city"
##############################################
print("T-tests for beach vs city on each topic:")
for t in range(num_topics):
    topic_col = f"Topic_{t}"

    # Subset the data by property type
    beach_values = topics_df.loc[topics_df['housetype'] == 'beach', topic_col]
    city_values = topics_df.loc[topics_df['housetype'] == 'city', topic_col]

    # Perform Welch's t-test (equal_var=False)
    stat, p_value = ttest_ind(beach_values, city_values, equal_var=False)

    print(f"{topic_col} - t-statistic: {stat:.4f}, p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("  --> Significant difference at alpha=0.05\n")
    else:
        print("  --> No significant difference at alpha=0.05\n")


T-tests for beach vs city on each topic:
Topic_0 - t-statistic: -1.6223, p-value: 0.1048
  --> No significant difference at alpha=0.05

Topic_1 - t-statistic: -1.5060, p-value: 0.1321
  --> No significant difference at alpha=0.05

Topic_2 - t-statistic: 6.2401, p-value: 0.0000
  --> Significant difference at alpha=0.05

Topic_3 - t-statistic: -1.9552, p-value: 0.0506
  --> No significant difference at alpha=0.05

Topic_4 - t-statistic: 0.1127, p-value: 0.9103
  --> No significant difference at alpha=0.05

Topic_5 - t-statistic: -3.1537, p-value: 0.0016
  --> Significant difference at alpha=0.05



In [ ]:
##############################################
# 2) T-tests for "Enes" vs. "Ryan"
##############################################
print("T-tests for Enes vs Ryan on each topic:")
for t in range(num_topics):
    topic_col = f"Topic_{t}"

    # Subset the data by creator
    enes_values = topics_df.loc[topics_df['creator'] == 'Enes', topic_col]
    ryan_values = topics_df.loc[topics_df['creator'] == 'Ryan', topic_col]

    # Perform Welch's t-test (equal_var=False)
    stat, p_value = ttest_ind(enes_values, ryan_values, equal_var=False)

    print(f"{topic_col} - t-statistic: {stat:.4f}, p-value: {p_value:.4f}")
    if p_value < 0.05:
        print("  --> Significant difference at alpha=0.05\n")
    else:
        print("  --> No significant difference at alpha=0.05\n")

T-tests for Enes vs Ryan on each topic:
Topic_0 - t-statistic: 1.6518, p-value: 0.0986
  --> No significant difference at alpha=0.05

Topic_1 - t-statistic: 0.3779, p-value: 0.7055
  --> No significant difference at alpha=0.05

Topic_2 - t-statistic: 4.7426, p-value: 0.0000
  --> Significant difference at alpha=0.05

Topic_3 - t-statistic: -3.7337, p-value: 0.0002
  --> Significant difference at alpha=0.05

Topic_4 - t-statistic: -3.5929, p-value: 0.0003
  --> Significant difference at alpha=0.05

Topic_5 - t-statistic: -0.1282, p-value: 0.8980
  --> No significant difference at alpha=0.05



## ANOVA Test

In [ ]:
!pip install statsmodels
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols


In [ ]:
# List of topic columns (assuming you have 6 topics)
topics = [f"Topic_{i}" for i in range(6)]

for topic in topics:
    # Build the formula: Topic proportion ~ property_type + creator + their interaction
    formula = f"{topic} ~ C(housetype) + C(creator) + C(housetype):C(creator)"

    # Fit the OLS model
    model = ols(formula, data=topics_df).fit()

    # Perform the ANOVA
    anova_table = sm.stats.anova_lm(model, typ=2)
    # typ=2 gives you Type II sums of squares, which is common in balanced or unbalanced designs.

    print(f"ANOVA for {topic}:")
    print(anova_table)
    print("\n")


ANOVA for Topic_0:
                            sum_sq      df         F    PR(>F)
C(housetype)              0.006371     1.0  3.670140  0.055439
C(creator)                0.006819     1.0  3.928152  0.047525
C(housetype):C(creator)   0.000358     1.0  0.206390  0.649627
Residual                 11.475915  6611.0       NaN       NaN


ANOVA for Topic_1:
                            sum_sq      df          F    PR(>F)
C(housetype)              0.004586     1.0   2.508361  0.113291
C(creator)                0.000746     1.0   0.408125  0.522945
C(housetype):C(creator)   0.023735     1.0  12.981776  0.000317
Residual                 12.086864  6611.0        NaN       NaN


ANOVA for Topic_2:
                            sum_sq      df          F        PR(>F)
C(housetype)              0.086857     1.0  30.441812  3.570728e-08
C(creator)                0.042888     1.0  15.031298  1.067675e-04
C(housetype):C(creator)   0.010788     1.0   3.780846  5.188481e-02
Residual                 18.8626



1.   Topic_0
**No significant difference in Topic_0 based on property_type or creator.**


2.   Topic_1

C(housetype) p ≈ 0.0825 → Not significant at the 5% level (but borderline).
C(creator) p ≈ 0.0232 → Significant.
Interaction p ≈ 0.210 → Not significant.
Interpretation: **There’s a significant difference in Topic_1 proportions by creator (Enes vs. Ryan**), but not by property_type (beach vs. city). Also, no interaction effect.

3.   Topic_2

C(housetype) has a small p‐value (likely < 0.001), so it’s significant.
C(creator) is borderline or not significant (p > 0.05).
Interaction is not significant.
Interpretation: **Topic_2 differs significantly between beach and city. No significant difference by creator**, nor an interaction.


4.   Topic_3
**No significant difference in Topic_3 proportions based on property_type or creator.**


5.   Topic_4

C(housetype) p > 0.05 → Not significant.
C(creator) p ≈ 0.0049 → Significant.
Interaction p ≈ 0.15 → Not significant.
**Topic_4 differs significantly by creator.** No difference by property_type or in the interaction.


6.   Topic_5
**No significant difference in Topic_5 based on property_type or creator.**

## Emojii detection

In [ ]:
!pip install emoji
import emoji
import re
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

# If you're in Colab (or Jupyter), enable inline plots:
%matplotlib inline


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 11.9 MB/s eta 0:00:00


In [ ]:
def extract_emojis(text):
    """
    Returns a list of all emoji characters found in 'text'.
    """
    return [char for char in text if char in emoji.EMOJI_DATA]


In [ ]:
all_emojis = []

for comment in df['comment'].astype(str):
    emojis_in_comment = extract_emojis(comment)
    all_emojis.extend(emojis_in_comment)


In [ ]:
emoji_counts = Counter(all_emojis)
# Print out the top 10 most common emojis
top_10_emojis = emoji_counts.most_common(10)
print("Top 10 Most Common Emojis:")
for e, c in top_10_emojis:
    print(f"{e}: {c}")


Top 10 Most Common Emojis:
🤍: 1136
❤: 486
😂: 340
👍: 143
👑: 135
😊: 122
😍: 103
🔥: 88
🙏: 84
👏: 81
